# 📘 SalesTeam AI — Explications Detaillees : Features, Modeles & Mecanismes de Recommandation

> Ce notebook explique en profondeur **comment fonctionne chaque composant** du systeme de recommandation SalesTeam AI :
> 1. Le role de `recency_days` et pourquoi c'est la feature #1
> 2. Comment les recommandations sont triees et affichees dans l'interface
> 3. Comment le Classifieur XGBoost trouve ses resultats
> 4. Comment le Regresseur XGBoost predit les quantites
> 5. L'importance et l'explication detaillee de chaque feature

---

# PARTIE 1 : Le Role de `recency_days`

## 1.1 C'est quoi `recency_days` ?

`recency_days` est simplement le **nombre de jours ecoules depuis la derniere commande** d'un client pour un produit donne.

### Calcul exact (dans `target_builder.py`) :
```python
last_date = pd.Timestamp(dates[-1])          # date du dernier achat
recency_days = (visit_date - last_date).days  # combien de jours depuis ?
```

### Exemple concret :
- Client `CLT091206` a commande du `REDMI BUDS 4 LITE` pour la derniere fois le **15 mars 2026**.
- Le commercial le visite le **25 avril 2026**.
- **`recency_days = 41 jours`**.

## 1.2 Pourquoi on l'a ajoute ?

### Avant (sans `recency_days`), le modele repondait a la question :
> *"Ce client a-t-il deja achete ce produit un jour dans sa vie ?"*

### Apres (avec `recency_days` + target visit-level), le modele repond a :
> *"Est-ce que ce client est **en train de revenir** acheter ce produit lors de cette visite ?"*

C'est toute la difference entre un **historique mort** et une **prediction vivante du moment de reassort**.

## 1.3 Pourquoi c'est devenu la feature #1 a 54.1% ?

### Avant la refonte visit-level :
`frequency` dominait a **56%** parce que le modele **trichait** :
- Negatif → `frequency = 0` → jamais achete
- Positif → `frequency > 0` → deja achete
- Le modele apprenait juste : `frequency > 0 → target = 1`. Pas d'intelligence.

### Apres la refonte visit-level :
Maintenant dans le dataset visit-level, **tous les candidats ont `frequency > 0`** (ce sont tous des produits deja achetes par le client). Donc `frequency` ne peut plus tricher.

Ce qui discrimine maintenant `target=1` (achete ce jour) vs `target=0` (skippe ce jour) c'est :

```
Est-ce que le client est "revenu a temps" pour ce produit ?

→ Si recency_days ≈ avg_delay_days  : il est dans son cycle normal  → probable reassort
→ Si recency_days >> avg_delay_days : en retard                     → tres probable reassort
→ Si recency_days << avg_delay_days : trop tot                      → pas encore besoin
```

**L'algorithme XGBoost l'a decouvert tout seul depuis les donnees : le timing predit le reachat.**

## 1.4 Le Trio de Features qui fait vraiment marcher l'IA

Ces 3 variables fonctionnent ensemble comme une **horloge de reassort** :

```
┌─────────────────────────────────────────────────────────┐
│         recency_days  =  41 jours                       │
│    /                                                     │
│         avg_delay_days  =  30 jours                      │
│    =                                                     │
│         recency_relative  =  1.37  (137% du cycle)       │
│                                                          │
│    → Le client est en RETARD de reassort de 37%          │
│    → L'IA predit target=1 avec haute probabilite         │
└─────────────────────────────────────────────────────────┘
```

Et par-dessus l'IA, le **business re-ranking** amplifie ce signal :

```
recency_relative >= 1.0  →  timing_boost = 2.0x
recency_relative >= 1.5  →  timing_boost = 3.0x

final_score = 0.71 x 2.0 x 1.0 = 1.42   ← remonte dans la liste
```

## 1.5 Le Flux Complet de Bout en Bout

Voici exactement ce qui se passe lors d'une requete de recommandation :

```
Commercial arrive chez client CLT091206
               |
[1] DATASET VISIT-LEVEL
    → On recupere les 30 produits historiques du client
    → On garde le snapshot le plus recent de chaque produit
    → On dedoublonne par code_article
               |
[2] CLASSIFIEUR XGBoost (10 features)
    Pour chaque produit :
    recency_days=41, avg_delay_days=30, recency_relative=1.37
    frequency=13, total_qty=980, trend=+0.15 ...
    → ml_score = 0.71 (71% de probabilite de reachat)
               |
[3] REGRESSEUR XGBoost (10 features)
    Pour les produits avec ml_score > 0.20 :
    last_qty=76, max_qty=120, std_qty=22 ...
    → quantite_suggeree = 76 unites
               |
[4] BUSINESS RE-RANKING
    recency_relative = 1.37 → timing_boost = 2.0x
    trend = +0.15 → trend_boost = 1.2x
    final_score = 0.71 x 2.0 x 1.2 = 1.704
    → Produit remonte en haut du classement
               |
[5] LLM LLAMA 3.3
    → Genere une explication en francais clair
    → "Le client commande ce produit tous les 30 jours
       et est actuellement en retard de reassort."
               |
[6] API FastAPI → Frontend React → Modal Premium
    → Commercial voit les N meilleurs produits tries
    → Clique sur une carte → Explication Llama s'ouvre
```

**En resume :** `recency_days` est devenu la feature la plus importante parce que c'est **la seule qui mesure le temps reel depuis le dernier achat**. Couplee a `avg_delay_days` (qui donne le rythme habituel), elle permet a l'IA de calculer precisement si un client est dans sa fenetre de reassort.

---

# PARTIE 2 : Comment les Recommandations Sont Triees et Affichees

## 2.1 Est-ce qu'il y a une liste infinie qu'on coupe a N ?

**Oui, exactement.** Pour chaque client, il existe autant de candidats que de produits qu'il a deja commandes historiquement. Par exemple, si `CLT070730` a achete 50 produits differents dans le passe, **le modele evalue les 50 en une seule fois**.

Le filtre se fait en **deux etapes** dans le backend :

```python
# Etape 1 : garder seulement les produits avec ML score >= 20%
candidates = ranked[ranked["probabilite_achat"] >= 0.20]

# Etape 2 : prendre les N premiers (N = ce que tu as choisi dans l'UI)
candidates = candidates.head(config.nb_suggestions)  # ex: head(15)
```

Donc :
- Si tu demandes **15** → on prend les **15 meilleurs** parmi tous les produits qualifies.
- Si tu demandes **5** → les **5 meilleurs**.
- Si tu demandes **2** → les **2 meilleurs**.

Ce sont toujours les `N` premiers du **meme classement**.

## 2.2 Comment ils sont tries — ET pourquoi les % semblent desordonnes

### Le tri se fait par `final_score`, PAS par le % affiche (`score_confiance`) !

```python
ranked = df_client.sort_values(by="final_score", ascending=False)
```

- Le **`%` affiche** sur la carte (`score_confiance`) c'est le **score brut ML** (probabilite pure sortie de XGBoost).
- L'**ordre d'affichage** est base sur `final_score = ML x timing_boost x trend_boost`.

### Exemple reel pour `CLT070730` :

| Position Affichee | `score_confiance` (%) | `timing_boost` | `trend_boost` | `final_score` (tri reel) |
|---|---|---|---|---|
| 1er | 68.4% | 3.0x | 1.2x | 68.4% x 3.0 x 1.2 = **2.46** |
| 3eme | 60.8% | 3.0x | 1.0x | 60.8% x 3.0 x 1.0 = **1.82** |
| 8eme | 61.3% | 1.0x | 1.0x | 61.3% x 1.0 x 1.0 = **0.61** |

### Pourquoi le 3eme (60.8%) est avant le 8eme (61.3%) ?
→ Parce que le 3eme produit a un **timing_boost de 3.0x** (le client est tres en retard pour ce produit), alors que le 8eme a un timing_boost de 1.0x (le client n'est pas en retard).

→ Le business re-ranking **priorise les produits dont le client a besoin MAINTENANT**, meme si leur probabilite ML pure est legerement inferieure.

---

# PARTIE 3 : Comment le Classifieur XGBoost Trouve ses Resultats

## 3.1 Qu'est-ce que le Classifieur fait ?

Le classifieur repond a une seule question binaire :

> **"Est-ce que le client va acheter ce produit lors de sa prochaine visite ? Oui (1) ou Non (0) ?"**

Il ne dit pas "combien" (ca c'est le regresseur). Il dit juste **"est-ce que oui ou non, avec quelle probabilite"**.

## 3.2 Comment il apprend (Phase d'Entrainement)

Pendant l'entrainement, XGBoost a recu **987 070 exemples historiques** (train set) de la forme :

```
Exemple 1 : Client A, Produit X, recency_days=15, frequency=8, ...  → target=1 (il a achete)
Exemple 2 : Client A, Produit Y, recency_days=90, frequency=3, ...  → target=0 (il a skippe)
Exemple 3 : Client B, Produit X, recency_days=28, frequency=12, ... → target=1 (il a achete)
...
```

XGBoost construit des **arbres de decision** sequentiels. Chaque arbre corrige les erreurs du precedent. Voici a quoi ressemble un arbre simplifie :

```
                    recency_days <= 35 ?
                   /                    \\
                 OUI                    NON
                  |                      |
     recency_relative <= 1.2?     frequency >= 5 ?
        /           \\                /         \\
      OUI           NON            OUI         NON
       |             |              |           |
   target=1      target=0       target=0    target=0
   (achete)     (skippe)       (skippe)    (skippe)
```

L'algorithme decouvre **automatiquement** les seuils optimaux en minimisant l'erreur de prediction sur les donnees d'entrainement.

## 3.3 Comment il predit (Phase d'Inference — lors de la requete API)

Quand un commercial demande les recommandations pour `CLT091206`, voici ce qui se passe :

### Etape 1 : Extraction des features
Pour chaque produit connu du client, on extrait les 10 features a partir du dataset :

```
Produit : REDMI BUDS 4 LITE BLACK
─────────────────────────────────────
frequency         = 13      (nombre total de commandes passees)
total_qty         = 980     (quantite totale achetee dans le passe)
avg_qty           = 75.4    (quantite moyenne par commande)
avg_delay_days    = 30      (delai moyen entre commandes)
recency_days      = 41      (jours depuis le dernier achat)
recency_relative  = 1.37    (ratio : 41/30 = 137% du cycle)
std_qty           = 22.1    (ecart-type de quantite — stabilite)
min_qty           = 20      (plus petite commande passee)
best_month        = 3       (mois avec le plus de ventes = mars)
avg_seasonal_coef = 1.08    (coefficient saisonnier moyen)
```

### Etape 2 : Passage dans les 24 arbres de decision
XGBoost fait passer ces 10 valeurs dans ses **24 arbres** (le modele s'est arrete a 24 arbres grace a l'early stopping).

Chaque arbre donne un "vote" ponderes. La somme de tous les votes est convertie en probabilite via la **fonction sigmoide** :

$$P(\text{achat}) = \sigma\left(\sum_{k=1}^{24} f_k(x)\right) = \frac{1}{1 + e^{-\text{somme des votes}}}$$

### Etape 3 : Resultat
Le classifieur sort une probabilite entre 0 et 1 :
- `probabilite_achat = 0.71` → **71% de chance que le client reachete ce produit**.
- Tous les produits du client sont evalues en parallele.
- Ceux avec une probabilite > 0.20 sont retenus comme candidats.

---

# PARTIE 4 : Comment le Regresseur XGBoost Predit les Quantites

## 4.1 Qu'est-ce que le Regresseur fait ?

Le regresseur repond a la question :

> **"SI le client achete ce produit, COMBIEN d'unites va-t-il commander ?"**

Il ne s'active que pour les produits deja filtres par le classifieur (ceux avec `probabilite_achat >= 0.20`).

## 4.2 Ses 10 Features d'Entree

Le regresseur utilise un jeu de features legerement different du classifieur, oriente vers les **volumes** :

```
avg_qty           = 75.4    → Moyenne historique des quantites commandees
std_qty           = 22.1    → Variabilite des quantites
min_qty           = 20      → Plus petite commande passee
max_qty           = 120     → Plus grosse commande passee
last_qty          = 76      → Quantite de la derniere commande
frequency         = 13      → Nombre total de commandes
recency_days      = 41      → Jours depuis le dernier achat
avg_delay_days    = 30      → Delai moyen entre commandes
current_month_coef= 1.10    → Coefficient saisonnier du mois actuel
avg_seasonal_coef = 1.08    → Coefficient saisonnier moyen
```

## 4.3 Comment il predit la quantite

### Logique interne simplifiee :
XGBoost Regressor apprend des patterns comme :

```
SI last_qty = 76 ET frequency >= 10 ET current_month_coef > 1.0
   → La prochaine commande sera probablement ~80 unites

SI last_qty = 5 ET max_qty = 10 ET std_qty < 3
   → Client stable, prochaine commande ~5-6 unites

SI last_qty = 200 ET std_qty = 150
   → Client tres variable, prediction moins fiable (~100-300)
```

### Resultat final :
```python
raw_qty = regressor.predict(X_reg)      # ex: 75.8
quantite_suggeree = round(raw_qty)       # ex: 76 unites
quantite_suggeree = max(1, quantite)     # minimum 1 unite
```

### Quand le regresseur n'est pas disponible :
Si le regresseur echoue ou n'est pas charge, le systeme utilise un **fallback** :
```python
quantite_suggeree = ceil(avg_qty)  # arrondi vers le haut de la moyenne historique
```

---

# PARTIE 5 : Importance et Explication Detaillee de Chaque Feature

## 5.1 Classement d'Importance du Classifieur (Resultats reels)

| Rang | Feature | Importance | Role dans la prediction |
|---|---|---|---|
| 1 | `recency_days` | **54.1%** | Jours depuis le dernier achat. C'est le signal principal. |
| 2 | `recency_relative` | **15.2%** | Ratio recency/delai habituel. Mesure si le client est en retard. |
| 3 | `avg_delay_days` | **6.4%** | Delai moyen entre commandes. Donne le rythme habituel du client. |
| 4 | `best_month` | **6.3%** | Mois de l'annee ou le produit se vend le plus. Signal saisonnier. |
| 5 | `frequency` | **4.9%** | Nombre total de commandes passees. Mesure la fidelite au produit. |
| 6 | `avg_seasonal_coef` | **3.9%** | Coefficient saisonnier moyen. Poids relatif de la saison dans les ventes. |
| 7 | `min_qty` | **2.7%** | Plus petite quantite commandee. Indicateur de commande minimale. |
| 8 | `std_qty` | **2.6%** | Ecart-type des quantites. Mesure la stabilite/variabilite du client. |
| 9 | `total_qty` | **2.1%** | Volume total achete historiquement. Mesure de la relation globale. |
| 10 | `avg_qty` | **1.8%** | Quantite moyenne par commande. Signal de volume habituel. |

## 5.2 Explication Detaillee de Chaque Feature

### `recency_days` — Jours depuis le dernier achat (54.1%)
- **Calcul** : `(date_visite - date_dernier_achat).days`
- **Pourquoi c'est important** : C'est la seule mesure du **temps reel**. Un client qui a achete hier n'a pas besoin de reacheter. Un client qui a achete il y a 2 mois est probablement en retard.
- **Valeur typique** : 1 a 365 jours.
- **Exemple** : `recency_days = 41` → Le client n'a pas achete depuis 41 jours.

---

### `recency_relative` — Ratio de cycle de reassort (15.2%)
- **Calcul** : `recency_days / avg_delay_days`
- **Pourquoi c'est important** : `recency_days` seul ne suffit pas. 41 jours c'est beaucoup si le client achete tous les 30 jours (`recency_relative = 1.37`), mais c'est peu s'il achete tous les 90 jours (`recency_relative = 0.46`).
- **Interpretation** :
  - `< 0.85` → Trop tot pour reacheter
  - `0.85 - 1.0` → Fenetre de reassort
  - `> 1.0` → En retard de reassort
  - `> 1.5` → Tres en retard

---

### `avg_delay_days` — Delai moyen entre commandes (6.4%)
- **Calcul** : Moyenne des ecarts (en jours) entre chaque paire de commandes consecutives.
- **Pourquoi c'est important** : Donne le **rythme naturel** du client pour ce produit. Un client qui commande tous les 15 jours est different d'un client qui commande tous les 90 jours.
- **Exemple** : Si un client a commande les 1er janvier, 1er fevrier, 1er mars → `avg_delay_days = 30`.

---

### `best_month` — Mois de pic saisonnier (6.3%)
- **Calcul** : Le mois de l'annee (1-12) ou la quantite moyenne commandee est la plus elevee.
- **Pourquoi c'est important** : Certains produits ont des pics saisonniers (ecouteurs en decembre pour les fetes, climatiseurs en ete). Le modele detecte si la visite tombe dans la periode favorable.
- **Exemple** : `best_month = 12` → Le produit se vend surtout en decembre.

---

### `frequency` — Nombre total de commandes (4.9%)
- **Calcul** : Nombre total de lignes d'achat dans l'historique pour cette paire (client, produit).
- **Pourquoi c'est important** : Mesure la **fidelite** du client a ce produit. Un produit commande 15 fois est un produit regulier ; un produit commande 1 seule fois est peut-etre un achat ponctuel.
- **Attention** : Dans le dataset visit-level, tous les candidats ont `frequency >= 1`, donc cette feature ne peut plus tricher.

---

### `avg_seasonal_coef` — Coefficient saisonnier moyen (3.9%)
- **Calcul** : Moyenne ponderee des coefficients saisonniers des mois ou le client a achete ce produit. Chaque mois a un coefficient predetermine (ex: decembre = 1.30, fevrier = 0.90).
- **Pourquoi c'est important** : Ajuste la prediction en fonction de la saisonnalite historique du produit pour ce client.

---

### `min_qty` — Plus petite quantite commandee (2.7%)
- **Calcul** : `min(toutes_les_quantites_passees)`
- **Pourquoi c'est important** : Si un client commande au minimum 50 unites a chaque fois, c'est un signal de volume eleve. Si `min_qty = 1`, c'est un achat occasionnel.

---

### `std_qty` — Ecart-type des quantites (2.6%)
- **Calcul** : Ecart-type statistique de toutes les quantites passees.
- **Pourquoi c'est important** : Mesure la **stabilite** du comportement d'achat.
  - `std_qty = 2` → Client tres stable, commande toujours la meme quantite.
  - `std_qty = 100` → Client imprevisible, commandes tres variables.

---

### `total_qty` — Volume total achete (2.1%)
- **Calcul** : `sum(toutes_les_quantites_passees)`
- **Pourquoi c'est important** : Mesure la **relation globale** entre le client et le produit. Un total de 5000 unites indique un partenariat fort.

---

### `avg_qty` — Quantite moyenne par commande (1.8%)
- **Calcul** : `total_qty / frequency`
- **Pourquoi c'est important** : Donne le **volume habituel** par transaction. Utile surtout pour le regresseur qui predit combien commander.

---

# PARTIE 6 : Verification et Inspection du Modele

Executez la cellule ci-dessous pour charger le modele entraine et verifier l'importance des features.

In [ ]:
import os, json, joblib
import pandas as pd
import matplotlib.pyplot as plt

model_path = '../src/models/classifier_lsat.joblib'
metadata_path = '../src/models/classifier_lsat_metadata.json'

if not os.path.exists(model_path):
    model_path = 'src/models/classifier_lsat.joblib'
    metadata_path = 'src/models/classifier_lsat_metadata.json'

if os.path.exists(model_path) and os.path.exists(metadata_path):
    model = joblib.load(model_path)
    with open(metadata_path, 'r', encoding='utf-8') as f:
        meta = json.load(f)
    
    feature_cols = meta.get('feature_columns', [])
    importances = model.feature_importances_
    
    fi_df = pd.DataFrame({
        'Feature': feature_cols,
        'Importance (%)': [round(imp * 100, 2) for imp in importances]
    }).sort_values('Importance (%)', ascending=False).reset_index(drop=True)
    
    print('=== IMPORTANCE DES FEATURES DU CLASSIFIEUR (VISIT-LEVEL) ===')
    print(fi_df.to_string(index=False))
    print()
    
    # Graphique
    fig, ax = plt.subplots(figsize=(10, 5))
    bars = ax.barh(fi_df['Feature'][::-1], fi_df['Importance (%)'][::-1], color='#1a56e8')
    ax.set_xlabel('Importance (%)')
    ax.set_title('Importance des Features — Classifieur XGBoost (Visit-Level)')
    plt.tight_layout()
    plt.show()
else:
    print('Artefacts du modele introuvables.')

In [ ]:
# Inspection du Regresseur
reg_path = '../src/models/regressor_lsat.joblib'
reg_meta_path = '../src/models/regressor_lsat_metadata.json'

if not os.path.exists(reg_path):
    reg_path = 'src/models/regressor_lsat.joblib'
    reg_meta_path = 'src/models/regressor_lsat_metadata.json'

if os.path.exists(reg_path) and os.path.exists(reg_meta_path):
    regressor = joblib.load(reg_path)
    with open(reg_meta_path, 'r', encoding='utf-8') as f:
        reg_meta = json.load(f)
    
    reg_features = reg_meta.get('feature_columns', [])
    reg_importances = regressor.feature_importances_
    
    reg_df = pd.DataFrame({
        'Feature': reg_features,
        'Importance (%)': [round(imp * 100, 2) for imp in reg_importances]
    }).sort_values('Importance (%)', ascending=False).reset_index(drop=True)
    
    print('=== IMPORTANCE DES FEATURES DU REGRESSEUR ===')
    print(reg_df.to_string(index=False))
    print()
    
    fig, ax = plt.subplots(figsize=(10, 5))
    bars = ax.barh(reg_df['Feature'][::-1], reg_df['Importance (%)'][::-1], color='#f59e0b')
    ax.set_xlabel('Importance (%)')
    ax.set_title('Importance des Features — Regresseur XGBoost')
    plt.tight_layout()
    plt.show()
else:
    print('Artefacts du regresseur introuvables.')